In [1]:
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras.losses import CategoricalCrossentropy

import matplotlib.pyplot as plt

import tqdm

import numpy as np
from mne.decoding import CSP
from config import Config

from network import NetCNN1D

from torch.utils.data import DataLoader

import os, sys

from mne.decoding import CSP
from sklearn.model_selection import StratifiedKFold
import tqdm

import numpy as np
import sys, os

import dataset_BCICIV2a
from config_BCICIV2a import Config

config = Config()




# Choose from: 'Left', 'Right', 'Foot' and 'Tongue'
config.used_classes = ['Right', 'Tongue']


config.t_start = -0.1
config.t_end = 3.0

config.n_csp_components = 3

2025-03-23 22:10:44.274785: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-23 22:10:44.274819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-23 22:10:44.275978: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-23 22:10:44.282330: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-23 22:10:45.138864: W tensorflow/compiler/tf2

In [2]:
subject_accs = []
subjects_std = []
for subject_id in tqdm.tqdm(range(1, 10)):

    if subject_id == 4:
        continue

    
    X, y = dataset_BCICIV2a.subject_dataset(config, subject_id)

    crossValidation_KF = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


    acc = []
    Y_preds = []

    shuffle_indices = np.random.permutation(len(y))

    
    X = X[shuffle_indices]
    y = y[shuffle_indices]

    for train_index, valid_index in crossValidation_KF.split(X,y):
        X_train = X[train_index]
        Y_train = y[train_index]
        X_valid = X[valid_index]
        Y_valid = y[valid_index]


        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

        csp = CSP(n_components=config.n_csp_components, reg=None, log=None, norm_trace=False, transform_into='csp_space')

        X_train = csp.fit_transform(X_train, Y_train)
        X_valid = csp.transform(X_valid)

        sys.stdout.close()
        sys.stdout = old_stdout


        X_train = X_train.transpose(0, 2, 1)
        X_valid = X_valid.transpose(0, 2, 1)

        Y_train = tf.keras.utils.to_categorical(Y_train, num_classes = 2)
        Y_valid = tf.keras.utils.to_categorical(Y_valid, num_classes = 2)



        model = NetCNN1D() 


        model.compile(
                loss= CategoricalCrossentropy(from_logits=True, label_smoothing=0.05),
                optimizer=keras.optimizers.Adam(learning_rate=1e-4, weight_decay = 2e-5), 
                metrics=[keras.metrics.CategoricalAccuracy(name='acc')]
        ) 



        history = model.fit(X_train, Y_train, validation_data=(X_valid, Y_valid), epochs=100, batch_size=16, verbose=0)

        
        #plt.figure(figsize=(15,5))
        #plt.title(f"subject = {subject_id} | N_examples = {X_train.shape[0]} | {max(history.history['val_acc'])} ")

        #plt.subplot(1,2,1)
        #plt.plot(history.history['loss'])
        #plt.plot(history.history['val_loss'])


        #plt.subplot(1,2,2)
        #plt.plot(history.history['acc'])
        #plt.plot(history.history['val_acc'])

        acc.append(max(history.history['val_acc']))



    subject_accs.append(np.mean(acc))
    subjects_std.append(np.std(acc))
    
    

print("\n\n ====================================")
for i in range(len(subject_accs)):
    subject_id = i+1
    if i >= 3:
        subject_id += 1 
    print(f"subject {subject_id} = {100*subject_accs[i]:.2f}% +- {100*subjects_std[i]:.1f}")

  0%|          | 0/9 [00:00<?, ?it/s]

Extracting EDF parameters from /home/antaeus/Neurobus/BCICIV_2a_gdf/A01T.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Not setting metadata
144 matching events found


/usr/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Setting baseline interval to [-0.1, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 72 events and 776 original time points ...
0 bad epochs dropped
Loading data for 72 events and 776 original time points ...
0 bad epochs dropped


2025-03-23 22:10:49.317380: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-03-23 22:10:49.318698: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
 11%|█         | 1/9 [00:30<04:01, 30.18s/it]

...

Extracting EDF parameters from /home/antaeus/Neurobus/BCICIV_2a_gdf/A09T.gdf...
GDF file detected
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['1023', '1072', '276', '277', '32766', '768', '769', '770', '771', '772']
Not setting metadata
144 matching events found
Setting baseline interval to [-0.1, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Loading data for 72 events and 776 original time points ...
0 bad epochs dropped
Loading data for 72 events and 776 original time points ...


/usr/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


0 bad epochs dropped


100%|██████████| 9/9 [04:07<00:00, 27.51s/it]



subject 1 = 98.60% +- 1.7
subject 2 = 62.54% +- 3.7
subject 3 = 93.77% +- 5.5
subject 5 = 69.43% +- 1.6
subject 6 = 60.42% +- 9.4
subject 7 = 86.72% +- 7.6
subject 8 = 89.53% +- 5.1
subject 9 = 89.63% +- 4.8
